# Ch 7 · Exercise 03 — Out-of-memory: driver vs executor  [local Spark]

Runs on local Apache Spark, **not** Databricks. It provokes the two out-of-memory (OOM) failures the exam
distinguishes — one on the **driver**, one in an **executor task** — and shows how each looks.

**Important: run ONE failure per kernel.** An OOM can leave the Spark session broken, so each of the two
failure cells says to **restart the kernel** before the next. Section 1 is the safe baseline; section 2 is the
driver OOM; section 3 is the task OOM. Do 1, then restart and do 2, then restart and do 3.

**Spill (Day 2) vs OOM (today):** spill is Spark's defense — an operation that *can* spill writes to disk and
finishes, slower. An OOM is what happens where spilling can't help: the driver holding too much, or one
partition too large for one task.

In [ ]:
# Run this first in every fresh kernel. driver.memory applies only at JVM startup.
DRIVER_MEMORY = "1g"

from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.master("local[*]").appName("oom-demo")
         .config("spark.driver.memory", DRIVER_MEMORY).getOrCreate())
sc = spark.sparkContext
def heap():
    rt = sc._jvm.java.lang.Runtime.getRuntime()
    return f"used ~{(rt.totalMemory()-rt.freeMemory())//(1024*1024):,} / max {rt.maxMemory()//(1024*1024):,} MB"
print("Spark", spark.version, "| cores", sc.defaultParallelism, "| heap", heap())
print("Spark UI:", sc.uiWebUrl)

## 1. The safe baseline — never risks memory

`count` runs on executors and returns one number; `show(5)` returns five rows however large the table is.
Neither pulls the whole dataset anywhere. This is the pattern to prefer.

In [ ]:
big = spark.range(200_000_000).withColumn("v", F.col("id")*2)
print("rows:", big.count())
big.show(5)
print("driver heap after count + show:", heap())      # barely moves

## 2. Driver OOM — `collect()` on a large result  ·  restart the kernel first

`collect()` brings **every** row to the driver, which is a single process no cluster can enlarge. The loop
collects growing slices so you can watch the driver's heap climb until a collect is too big to fit.

**Read the result carefully.** When the driver's JVM runs out of heap, Python often can't see the real error —
it reports a broken connection (`ConnectionRefused`, `Error while sending or receiving`) instead. **The true
cause, `OutOfMemoryError: Java heap space`, is in the notebook's log output and the terminal**, not always in
the message Python catches. That gap is itself the lesson: on Databricks you confirm a driver OOM from the
**cluster event log and driver logs**, not from the Python traceback alone.

In [ ]:
# Restart the kernel, then run the setup cell, then this.
big = spark.range(200_000_000).withColumn("v", F.col("id")*2)
for n in (1_000_000, 5_000_000, 20_000_000, 80_000_000, 200_000_000):
    try:
        rows = big.limit(n).collect()
        print(f"  collected {n:>12,} rows   driver heap {heap()}")
        rows = None
    except Exception as ex:
        print(f"  collecting {n:,} rows FAILED in the DRIVER (expected).")
        print("   Python saw:", type(ex).__name__, "-", (str(ex).splitlines() or [''])[0][:120])
        print("   The real cause (OutOfMemoryError) is in the log/terminal above.")
        break

## 3. Executor / task OOM — one partition too large  ·  restart the kernel first

Every row is forced onto one key, then gathered into a single in-memory list that can't be spilled, so it must
fit in **one task's** memory. On a small heap it can't, and a **task** fails with `OutOfMemoryError` — visible
in the Spark UI's failed stage, and in the log. As in section 2, the Python message may show a broken
connection rather than the OOM itself.

In [ ]:
# Restart the kernel, then run the setup cell, then this.
try:
    (spark.range(50_000_000)
        .withColumn("k", F.lit(1))
        .withColumn("payload", F.repeat(F.lit("x"), 100))
        .groupBy("k").agg(F.collect_list("payload").alias("all"))
        .write.format("noop").mode("overwrite").save())
    print("completed — did not fail; try a larger range or smaller DRIVER_MEMORY")
except Exception as ex:
    print("A TASK FAILED (expected).")
    print("  Python saw:", type(ex).__name__, "-", (str(ex).splitlines() or [''])[0][:120])
    print("  Look in the Spark UI (Stages: the failed stage) and the log for OutOfMemoryError.")

## What to check
- **Section 1** stays cheap: `count` and `show(5)` barely move the driver's heap, whatever the table's size.
- **Section 2** exhausts the **driver** — the `collect()` pattern that, on Databricks, makes a cluster report
  *the spark driver has stopped unexpectedly* or *driver is up but not responsive, likely due to GC*.
- **Section 3** exhausts a **task** — one partition too large, with no spill possible; more partitions or less
  skew would relieve it.
- **The diagnosis lesson:** the Python traceback often shows only a broken connection. The real
  `OutOfMemoryError` is in the driver log and the Spark UI. On Databricks, that means the **cluster event log**
  and **driver logs** — the subject of the theory half of this exercise.

Driver OOM: fix the *code* (don't `collect` large data) or raise driver memory. Task OOM: fix the *data layout*
(more partitions, less skew) or raise executor memory.